In [ ]:
c!git clone https://github.com/google-research/big_vision.git


Cloning into 'big_vision'...


In [ ]:
from __future__ import annotations
import os, sys, json, random
from dataclasses import dataclass
from typing import List, Tuple
import re

import cv2
import numpy as np
import tensorflow as tf
from PIL import Image, ImageDraw


@dataclass
class Config:
    data_path: str = r"D:\Harisankar\MSc. Remote Sensing and Geoinformatics\Internship2 @ DLR\new data for journal\scene_clip_building\scene_clip_building"
    images_folder_name: str = "images"          # RGBs
    masks_folder_name: str = "masks"            # binary/grayscale masks
    output_jsonl_name: str = "buildings_segm.jsonl"  # single JSONL

    # If mask filenames differ from images, you can remap:
    # e.g., replace "_mask.png" -> ".png" when looking for the image
    mask_to_image_replace_from: str = ""        # set to "_mask.png" if needed
    mask_to_image_replace_to: str = ""          # set to ".png" if needed

    prefix: str = "segment buildings"
    class_name: str = "Building"

    # Contour/mask knobs
    threshold: int = 127      # binarize masks (0..255)
    epsilon: float = 0.001    # Douglas–Peucker simplification factor
    npoints: int = 8          # min vertices after simplification
    min_area: int = 36        # min contour area (px)

    # Optional: write quick bbox previews (no decoder) for sanity
    make_preview: bool = True
    preview_count: int = 8
    preview_subdir: str = "preview_bbox"
    overlay_alpha: float = 0.45

cfg = Config()
# Big Vision repo path (contains 'big_vision' package)
cfg.big_vision_repo = r"D:\Harisankar\MSc. Remote Sensing and Geoinformatics\Internship2 @ DLR\new data for journal\big_vision"

#  ENV & IMPORTS
random.seed(123)
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

if cfg.big_vision_repo and cfg.big_vision_repo not in sys.path:
    sys.path.append(cfg.big_vision_repo)

try:
    from big_vision.pp.proj.paligemma.segmentation import (
        encode_to_codebook_indices,
        get_checkpoint,
    )
except Exception as e:
    raise RuntimeError(
        "Could not import Paligemma helpers. "
        f"Ensure '{cfg.big_vision_repo}' points to the Big Vision repo root."
    ) from e

CHECKPOINT = get_checkpoint(model="oi")  # OI tokenizer (K=128)

#  HELPERS
def _reduce_contours(contours, epsilon: float):
    out = []
    for cnt in contours:
        per = cv2.arcLength(cnt, True)
        out.append(cv2.approxPolyDP(cnt, max(0.0, epsilon) * per, True))
    return out

def _mask_from_contour(h: int, w: int, cnt) -> np.ndarray:
    m = np.zeros((h, w), np.uint8)
    cv2.drawContours(m, [cnt], 0, 255, cv2.FILLED)
    return m > 0

def _bbox_xyxy(cnt) -> Tuple[int, int, int, int]:
    x, y, w, h = cv2.boundingRect(cnt)
    return x, y, x + w, y + h

def _format_loc_tokens(y1, x1, y2, x2, H, W) -> str:
    loc_tokens = tf.constant([f"<loc{i:04d}>" for i in range(1024)])
    bbox = np.array([y1, x1, y2, x2], np.float32) / np.array([H, W, H, W], np.float32)
    binned = tf.cast(tf.round(bbox * 1023.0), tf.int32)
    binned = tf.clip_by_value(binned, 0, 1023)
    return tf.strings.reduce_join(tf.gather(loc_tokens, binned)).numpy().decode("utf-8")

def _format_seg_tokens(boolean_mask: np.ndarray, y1, x1, y2, x2) -> str:
    seg_tokens = tf.constant([f"<seg{i:03d}>" for i in range(128)])
    yy1, xx1, yy2, xx2 = map(int, map(round, (y1, x1, y2, x2)))
    crop = boolean_mask[yy1:yy2, xx1:xx2]
    if crop.size == 0:
        crop = np.zeros((1, 1), np.uint8)
    t = tf.convert_to_tensor(crop.astype(np.uint8))[None, ..., None]   # [1,h,w,1]
    t = tf.image.resize(t, [64, 64], method="bilinear", antialias=True)
    idx = encode_to_codebook_indices(CHECKPOINT, t)[0]                 # [4,4]
    return tf.strings.reduce_join(tf.gather(seg_tokens, idx)).numpy().decode("utf-8")

def _mask_to_row(mask_path: str, images_dir: str) -> dict:
    mask_name = os.path.basename(mask_path)
    img_name = mask_name.replace(cfg.mask_to_image_replace_from, cfg.mask_to_image_replace_to)
    row = {"image": img_name, "prefix": cfg.prefix, "suffix": ""}

    m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if m is None:
        return row
    H, W = m.shape[:2]
    _, mb = cv2.threshold(m, cfg.threshold, 255, cv2.THRESH_BINARY)
    if np.count_nonzero(mb) == 0:
        return row

    contours, _ = cv2.findContours(mb, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = _reduce_contours(contours, cfg.epsilon)

    kept = []
    for c in contours:
        if len(c) < max(3, cfg.npoints):    # why: spurious crumbs
            continue
        if cv2.contourArea(c) < float(cfg.min_area):
            continue
        kept.append(c)
    if not kept:
        kept = contours

    chunks = []
    bm = None  # postpone boolean mask until we need it
    for cnt in kept:
        x1, y1, x2, y2 = _bbox_xyxy(cnt)
        if bm is None:
            bm = _mask_from_contour(H, W, cnt)
        else:
            bm = _mask_from_contour(H, W, cnt)  # why: each contour separately

        loc = _format_loc_tokens(y1, x1, y2, x2, H, W)
        seg = _format_seg_tokens(bm, y1, x1, y2, x2)
        chunks.append(f"{loc}{seg} {cfg.class_name}")

    row["suffix"] = " ; ".join(chunks) if chunks else ""
    return row

def _write_jsonl(rows: List[dict], path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

def _quick_preview(rows: List[dict], images_dir: str, out_dir: str, n: int, alpha: float) -> None:
    os.makedirs(out_dir, exist_ok=True)
    pick = rows[:n]
    for r in pick:
        img_path = os.path.join(images_dir, r["image"])
        if not os.path.exists(img_path):
            continue
        im = Image.open(img_path).convert("RGB")
        W, H = im.size
        overlay = Image.new("L", (W, H), 0)
        parts = [s for s in r.get("suffix","").split(" ; ") if s.strip()]
        for p in parts:
            # find all <loc####> tokens even if concatenated
            loc_ids = [int(m) for m in re.findall(r"<loc(\d{4})>", p)]
            if len(loc_ids) < 4:
                continue
            # decode 4 loc codes → bbox
            pts = []
            for code in loc_ids[:4]:
                qy, qx = divmod(code, 1024)    # (row=y, col=x) in 1024 bins
                y = int(round(qy / 1023 * (H - 1)))
                x = int(round(qx / 1023 * (W - 1)))
                pts.append((x, y))
            xs, ys = [p[0] for p in pts], [p[1] for p in pts]
            x1, x2 = min(xs), max(xs); y1, y2 = min(ys), max(ys)
            ImageDraw.Draw(overlay).rectangle([x1, y1, x2, y2], fill=255)
        color = Image.new("RGB", (W, H), (255, 0, 0))
        comp = Image.composite(color, im, overlay)
        comp = Image.blend(im, comp, alpha)
        comp.save(os.path.join(out_dir, r["image"]))

#  RUN
images_dir = os.path.join(cfg.data_path, cfg.images_folder_name)
masks_dir  = os.path.join(cfg.data_path, cfg.masks_folder_name)
out_jsonl  = os.path.join(cfg.data_path, cfg.output_jsonl_name)

mask_files = sorted([os.path.join(masks_dir, f) for f in os.listdir(masks_dir)
                     if not f.startswith(".")])

rows: List[dict] = []
for mp in mask_files:
    rows.append(_mask_to_row(mp, images_dir))

_write_jsonl(rows, out_jsonl)
print(f"Wrote JSONL: {out_jsonl}  ({len(rows)} rows)")

if cfg.make_preview:
    _quick_preview(
        rows, images_dir,
        os.path.join(cfg.data_path, cfg.preview_subdir),
        n=min(cfg.preview_count, len(rows)),
        alpha=cfg.overlay_alpha,
    )
    print("Saved quick previews.")

Wrote JSONL: D:\Harisankar\MSc. Remote Sensing and Geoinformatics\Internship2 @ DLR\new data for journal\scene_clip_building\scene_clip_building\buildings_segm.jsonl  (2070 rows)
Saved quick previews.


In [ ]:
import functools
import re
import numpy as np
import PIL.Image
import cv2
import jax
import jax.numpy as jnp
import flax.linen as nn
# import torch
import supervision as sv
import os, sys, json, random

# 1. VQ-VAE Decoder Setup

_MODEL_PATH = r"D:\Harisankar\MSc. Remote Sensing and Geoinformatics\Internship2 @ DLR\new data for journal\vae-oid (1).npz"

def _get_params(checkpoint):
    """Convert PyTorch checkpoint weights to Flax params."""
    def transp(kernel): return np.transpose(kernel, (2, 3, 1, 0))
    def conv(name):
        return {"bias": checkpoint[name + ".bias"],
                "kernel": transp(checkpoint[name + ".weight"])}
    def resblock(name):
        return {"Conv_0": conv(name + ".0"),
                "Conv_1": conv(name + ".2"),
                "Conv_2": conv(name + ".4")}
    return {
        "_embeddings": checkpoint["_vq_vae._embedding"],
        "Conv_0": conv("decoder.0"),
        "ResBlock_0": resblock("decoder.2.net"),
        "ResBlock_1": resblock("decoder.3.net"),
        "ConvTranspose_0": conv("decoder.4"),
        "ConvTranspose_1": conv("decoder.6"),
        "ConvTranspose_2": conv("decoder.8"),
        "ConvTranspose_3": conv("decoder.10"),
        "Conv_1": conv("decoder.12"),
    }

def _quantized_values_from_codebook_indices(codebook_indices, embeddings):
    batch_size, num_tokens = codebook_indices.shape
    assert num_tokens == 16, codebook_indices.shape
    unused_num_embeddings, embedding_dim = embeddings.shape
    encodings = jnp.take(embeddings, codebook_indices.reshape((-1)), axis=0)
    encodings = encodings.reshape((batch_size, 4, 4, embedding_dim))
    return encodings

@functools.cache
def _get_reconstruct_masks():
    """Return function that decodes <seg> tokens -> 64x64 mask patch."""

    class ResBlock(nn.Module):
        features: int
        @nn.compact
        def __call__(self, x):
            orig = x
            x = nn.Conv(self.features, (3, 3), padding=1)(x)
            x = nn.relu(x)
            x = nn.Conv(self.features, (3, 3), padding=1)(x)
            x = nn.relu(x)
            x = nn.Conv(self.features, (1, 1))(x)
            return x + orig

    class Decoder(nn.Module):
        @nn.compact
        def __call__(self, x):
            dim = 128
            x = nn.Conv(dim, (1, 1))(x)
            x = nn.relu(x)
            for _ in range(2):
                x = ResBlock(dim)(x)
            for _ in range(4):
                x = nn.ConvTranspose(
                    dim, (4, 4), strides=(2, 2),
                    padding=2, transpose_kernel=True)(x)
                x = nn.relu(x)
                dim //= 2
            x = nn.Conv(1, (1, 1))(x)
            return x

    def reconstruct_masks(codebook_indices):
        quantized = _quantized_values_from_codebook_indices(
            codebook_indices, params["_embeddings"])
        return Decoder().apply({"params": params}, quantized)

    with open(_MODEL_PATH, "rb") as f:
        params = _get_params(dict(np.load(f)))
    return jax.jit(reconstruct_masks, backend="cpu")

# 2. Token Parsing

_SEGMENT_DETECT_RE = re.compile(
    r"(.*?)" +
    r"<loc(\d{4})>" * 4 + r"\s*" +
    "(?:%s)?"
    % (r"<seg(\d{3})>" * 16) +
    r"\s*([^;<>]+)? ?(?:; )?",
)

def extract_objs(text, width, height, unique_labels=False):
    """Decode Paligemma text output into objs with bbox + mask."""
    objs, seen = [], set()
    while text:
        m = _SEGMENT_DETECT_RE.match(text)
        if not m:
            break
        gs = list(m.groups())
        before = gs.pop(0)
        name = gs.pop()
        y1, x1, y2, x2 = [int(x) / 1024 for x in gs[:4]]
        y1, x1, y2, x2 = map(round, (y1 * height, x1 * width,
                                     y2 * height, x2 * width))
        seg_indices = gs[4:20]
        if seg_indices[0] is None:
            mask = None
        else:
            seg_indices = np.array([int(x) for x in seg_indices], dtype=np.int32)
            m64, = _get_reconstruct_masks()(seg_indices[None])[..., 0]
            m64 = np.clip(np.array(m64) * 0.5 + 0.5, 0, 1)
            m64 = PIL.Image.fromarray((m64 * 255).astype("uint8"))
            mask = np.zeros([height, width])
            if y2 > y1 and x2 > x1:
                mask[y1:y2, x1:x2] = np.array(
                    m64.resize([x2 - x1, y2 - y1])) / 255.0
        content = m.group()
        if before:
            objs.append(dict(content=before))
            content = content[len(before):]
        while unique_labels and name in seen:
            name = (name or "") + "'"
        seen.add(name)
        objs.append(dict(content=content,
                         xyxy=(x1, y1, x2, y2),
                         mask=mask, name=name))
        text = text[len(before) + len(content):]
    if text:
        objs.append(dict(content=text))
    return objs

# 3. Convert to Supervision

def from_paligemma(text: str, resolution_wh: tuple[int, int], classes: list[str]) -> sv.Detections:
    w, h = resolution_wh
    results = extract_objs(text, w, h)
    xyxy, masks, class_id, class_name = [], [], [], []
    for r in results:
        xyxy.append(r["xyxy"])
        if r["mask"] is not None:
            _, m = cv2.threshold(r["mask"], 0.5, 1.0, cv2.THRESH_BINARY)
            masks.append(m)
        else:
            masks.append(np.zeros((h, w)))
        cls = r["name"].strip() if r["name"] else "building"
        class_id.append(classes.index(cls))
        class_name.append(cls)
    detections = sv.Detections(
        xyxy=np.array(xyxy).astype(int),
        mask=np.array(masks).astype(bool),
        class_id=np.array(class_id).astype(int))
    detections["class_name"] = class_name
    return detections

def visualize_jsonl(jsonl_path, images_dir, out_dir, n=200, classes=["Building"]):
    os.makedirs(out_dir, exist_ok=True)

    with open(jsonl_path, "r", encoding="utf-8") as f:
        rows = [json.loads(line) for line in f]

    random.shuffle(rows)  # <-- pick random order
    for i, row in enumerate(rows[:n]):
        img_path = os.path.join(images_dir, row["image"])
        if not os.path.exists(img_path):
            print(" Missing image:", img_path)
            continue

        image = PIL.Image.open(img_path).convert("RGB")
        w, h = image.size

        decoded_text = row.get("suffix", "").strip()
        if not decoded_text:   # skip if no tokens
            print(f"Skipping {row['image']} (empty suffix)")
            continue

        detections = from_paligemma(decoded_text, (w, h), classes)

        if detections.xyxy.shape[0] == 0:   # skip if no decoded objects
            print(f" Skipping {row['image']} (no objects decoded)")
            continue

        annotated = image.copy()
        annotated = sv.MaskAnnotator().annotate(annotated, detections)
        annotated = sv.LabelAnnotator(text_position=sv.Position.CENTER,
                                      smart_position=True).annotate(annotated, detections)

        out_file = os.path.join(out_dir, f"preview_{i}_{row['image']}")
        annotated.save(out_file)
        print(f" Saved preview: {out_file}")

    print(f"\nDone. {n} previews saved to {out_dir}")


#  Example usage
jsonl_path  = r"D:\Harisankar\MSc. Remote Sensing and Geoinformatics\Internship2 @ DLR\new data for journal\scene_clip_building\scene_clip_building\buildings_segm.jsonl"
images_dir  = r"D:\Harisankar\MSc. Remote Sensing and Geoinformatics\Internship2 @ DLR\new data for journal\scene_clip_building\scene_clip_building\images"
out_dir     = r"D:\Harisankar\MSc. Remote Sensing and Geoinformatics\Internship2 @ DLR\new data for journal\scene_clip_building\scene_clip_building\segm_preview"

visualize_jsonl(jsonl_path, images_dir, out_dir, n=200, classes=["Building"])

 Saved preview: D:\Harisankar\MSc. Remote Sensing and Geoinformatics\Internship2 @ DLR\new data for journal\scene_clip_building\scene_clip_building\segm_preview\preview_0_000626.png
 Saved preview: D:\Harisankar\MSc. Remote Sensing and Geoinformatics\Internship2 @ DLR\new data for journal\scene_clip_building\scene_clip_building\segm_preview\preview_1_001448.png
 Saved preview: D:\Harisankar\MSc. Remote Sensing and Geoinformatics\Internship2 @ DLR\new data for journal\scene_clip_building\scene_clip_building\segm_preview\preview_2_001729.png
 Saved preview: D:\Harisankar\MSc. Remote Sensing and Geoinformatics\Internship2 @ DLR\new data for journal\scene_clip_building\scene_clip_building\segm_preview\preview_3_001536.png
 Saved preview: D:\Harisankar\MSc. Remote Sensing and Geoinformatics\Internship2 @ DLR\new data for journal\scene_clip_building\scene_clip_building\segm_preview\preview_4_001514.png
 Saved preview: D:\Harisankar\MSc. Remote Sensing and Geoinformatics\Internship2 @ DLR\new

In [ ]:
pip install supervision

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip
